# EnergyDataModel — Quickstart

`energydatamodel` (EDM) is a typed, tree-shaped data model for describing energy systems — assets, areas, interconnections, and the time series attached to each. Every structural class ultimately inherits from a single `Element` root, with three sibling subtrees and one cross-cutting mixin:

- **`Node`** — graph vertices (equipment, areas, sensors, grid topology points). Has `members` and `tz`. Subclassed by `NodeAsset`, `GridNode`, `Sensor`, and `Area`.
- **`Edge`** — a directed relationship between two nodes (`Line`, `Transformer`, `Interconnection`, `Pipe`), via `EdgeAsset`.
- **`Collection`** — logical groupings (`Portfolio`, `Site`, `Region`, ...). Has `members` and `tz`. Not a graph vertex.
- **`Asset`** — cross-cutting mixin marking physical equipment (`commissioning_date`). Mixed into `NodeAsset` / `EdgeAsset`.

This notebook walks through:

1. Building a small portfolio tree.
2. Attaching time-series descriptors via the energy vocabulary.
3. Tree walking (`children()`, `to_tree()`).
4. Cross-tree references with `Reference[T]`.
5. Full JSON round-trip.
6. Tour of the powergrid + area types.

In [1]:
import json

import energydatamodel as edm
from shapely.geometry import Point, Polygon
from timedatamodel import DataType, Frequency

## 1. Build a tree

Start from the leaves (turbines) and aggregate upward. Every `Node` takes a `members` list of child entities. Use `geometry=Point(lon, lat)` for locations.

In [2]:
t01 = edm.WindTurbine(name="T01", capacity=3.5, hub_height=80, geometry=Point(12.78, 55.51))
t02 = edm.WindTurbine(name="T02", capacity=3.5, hub_height=80, geometry=Point(12.79, 55.52))

lillgrund = edm.WindFarm(name="Lillgrund", capacity=110, members=[t01, t02])

pv_array = edm.PVArray(capacity=5.0, surface_tilt=25, surface_azimuth=180)
rooftop = edm.PVSystem(name="Rooftop-01", capacity=5.0, surface_tilt=25, surface_azimuth=180, members=[pv_array])

site = edm.Site(name="Sweden-Offshore", geometry=Point(12.8, 55.5), members=[lillgrund, rooftop])

### Other asset types

Beyond wind/solar, EDM ships `Battery`, `HeatPump`, `Building`, `House`, `Reservoir`/`HydroTurbine`/`HydroPowerPlant`, and weather sensors. `Building` and `House` are `Asset`s that also contain child entities via `members`. Locations are set via `geometry=Point(lon, lat)`.

In [3]:
from zoneinfo import ZoneInfo

hp = edm.HeatPump(name="HP1", capacity=8, cop=3.5, energy_source="electricity",
                  timeseries=[edm.heating_demand(unit="kW")])
bat = edm.Battery(name="Bat1", storage_capacity=10, max_charge=5, max_discharge=5)

house = edm.House(
    name="House-42",
    geometry=Point(11.97, 57.71),
    tz=ZoneInfo("Europe/Stockholm"),
    members=[hp, bat],
)
print(house.to_tree())
print()
print("has_battery:", house.has_battery(), " batteries:", [b.name for b in house.get_batteries()])

House('House-42')
├── HeatPump('HP1')
└── Battery('Bat1')

has_battery: True  batteries: ['Bat1']


## 2. Energy vocabulary

Instead of subclassing `TimeSeries` (the old `ElectricityDemand(TimeSeries)` pattern), build `TimeSeriesDescriptor`s with convenience constructors. Each returns a descriptor whose `name` field is a dotted string like `electricity.supply` or `price.spot`.

Under the hood `build_metric(Quantity, Kind, Scope)` composes these names; constructors are thin wrappers.

In [4]:
# Attach time series to assets.
t01.timeseries = [
    edm.electricity_supply(unit="MW", data_type=DataType.FORECAST, frequency=Frequency.PT1H),
]
rooftop.timeseries = [
    edm.electricity_supply(unit="kW", data_type=DataType.ACTUAL),
]

# Peek at a descriptor.
t01.timeseries[0]

TimeSeriesDescriptor(name='electricity.supply', unit='MW', data_type=<DataType.FORECAST: 'FORECAST'>, timeseries_type=<TimeSeriesType.FLAT: 'FLAT'>, frequency=<Frequency.PT1H: 'PT1H'>, timezone='UTC', description=None)

In [5]:
# Or build custom metrics manually.
from energydatamodel import Kind, Quantity, Scope, build_metric

build_metric(Quantity.ELECTRICITY, Kind.DEMAND, Scope.AREA)

'electricity.demand.area'

## 3. Areas + Edges + cross-tree references

`Area(Node)` represents bidding zones, control areas, countries, weather cells — each as a proper subclass (`BiddingZone`, `Country`, etc.), distinguished by `isinstance`.

`Interconnection(Edge)` connects two areas. Because the connected nodes live elsewhere in the tree, we reference them by path via `Reference[T]` — the reference holds a path string until the tree is loaded, then resolves to the object.

In [6]:
se4 = edm.BiddingZone(name="SE4", timeseries=[edm.spot_price(unit="EUR / MWh")])
dk2 = edm.BiddingZone(name="DK2", timeseries=[edm.spot_price(unit="EUR / MWh")])

icx = edm.Interconnection(
    name="SE4-DK2",
    from_entity=edm.Reference("SE4"),
    to_entity=edm.Reference("DK2"),
    capacity_forward=1700,
    capacity_backward=1300,
    timeseries=[edm.cross_border_flow(unit="MW")],
)

portfolio = edm.Portfolio(name="Nordic", members=[site, se4, dk2, icx])
print(portfolio.to_tree())

Portfolio('Nordic')
├── Site('Sweden-Offshore')
│   ├── WindFarm('Lillgrund')
│   │   ├── WindTurbine('T01')
│   │   └── WindTurbine('T02')
│   └── PVSystem('Rooftop-01')
│       └── PVArray()
├── BiddingZone('SE4')
├── BiddingZone('DK2')
└── Interconnection('SE4-DK2')


### Geospatial: polygons on areas and regions

`Area`, `Region`, `SolarPowerArea`, `WindPowerArea` carry a Shapely `Polygon` (or `GeoMultiPolygon`) on a `geometry` / `geopolygon` field. Derived values come straight from Shapely (`.area`, `.bounds`, `.centroid`). Geometry survives JSON round-trip via GeoJSON.

In [7]:

# Rough bounding polygons for SE4 and DK2 (lon, lat).
se4.geometry = Polygon([(12.5, 55.0), (16.5, 55.0), (16.5, 58.0), (12.5, 58.0)])
dk2.geometry = Polygon([(11.0, 54.5), (12.8, 54.5), (12.8, 56.0), (11.0, 56.0)])

region = edm.Region(
    name="Scandinavia-South",
    geometry=Polygon([(8, 54), (17, 54), (17, 60), (8, 60)]),
)

print("SE4 area    :", se4.geometry.area)
print("SE4 bounds  :", se4.geometry.bounds)
print("SE4 centroid:", (se4.geometry.centroid.x, se4.geometry.centroid.y))

# Round-trip preserves the Polygon type (via GeoJSON under the hood).
geo_portfolio = edm.Portfolio(name="Nordic-Geo", members=[se4, dk2, region])
restored = edm.Portfolio.from_json(geo_portfolio.to_json())
assert isinstance(restored.members[0].geometry, Polygon)
assert restored.members[0].geometry.equals(se4.geometry)
print("round-trip preserved:", type(restored.members[0].geometry).__name__)

SE4 area    : 12.0
SE4 bounds  : (12.5, 55.0, 16.5, 58.0)
SE4 centroid: (14.5, 56.5)
round-trip preserved: Polygon


## 4. Tree walking

Every entity exposes a `children()` iterator. Recursive walks are trivial.

In [8]:
def walk(entity, depth=0):
    print("  " * depth + f"{type(entity).__name__}({entity.name!r})")
    for c in entity.children():
        walk(c, depth + 1)

walk(portfolio)

Portfolio('Nordic')
  Site('Sweden-Offshore')
    WindFarm('Lillgrund')
      WindTurbine('T01')
      WindTurbine('T02')
    PVSystem('Rooftop-01')
      PVArray(None)
  BiddingZone('SE4')
  BiddingZone('DK2')
  Interconnection('SE4-DK2')


In [9]:
# Generic tree reconstruction via add_child() — symmetric to children().
rebuilt = edm.Portfolio(name="Rebuilt")
rebuilt.add_child(edm.Site(name="Sweden-Offshore"))
rebuilt.add_child(edm.BiddingZone(name="SE4"))
print(rebuilt.to_tree())

Portfolio('Rebuilt')
├── Site('Sweden-Offshore')
└── BiddingZone('SE4')


## 5. JSON round-trip

`to_json()` emits a JSON-compatible dict. `from_json()` does a two-pass walk: first instantiate each entity via a class registry, then resolve every `Reference` against the tree root.

`_id` is excluded by default (use `include_ids=True` to emit it) — `_id` is a boundary field for downstream systems like `energydb`.

In [10]:
js = portfolio.to_json()
print(json.dumps(js, indent=2, default=str))

{
  "type": "Portfolio",
  "name": "Nordic",
  "members": [
    {
      "type": "Site",
      "name": "Sweden-Offshore",
      "geometry": {
        "__geometry__": true,
        "type": "Point",
        "coordinates": [
          12.8,
          55.5
        ]
      },
      "members": [
        {
          "type": "WindFarm",
          "name": "Lillgrund",
          "members": [
            {
              "type": "WindTurbine",
              "name": "T01",
              "timeseries": [
                {
                  "__type__": "TimeSeriesDescriptor",
                  "name": "electricity.supply",
                  "unit": "MW",
                  "data_type": "FORECAST",
                  "timeseries_type": "FLAT",
                  "frequency": "PT1H",
                  "timezone": "UTC",
                  "description": null
                }
              ],
              "geometry": {
                "__geometry__": true,
                "type": "Point",
                "c

In [11]:
# Round-trip.
restored = edm.Portfolio.from_json(js)

# Byte-equal after re-dump.
assert json.dumps(js, default=str) == json.dumps(restored.to_json(), default=str)

# References are resolved — from_entity/to_entity now point at the actual BiddingZone objects.
icx_restored = next(m for m in restored.members if isinstance(m, edm.Interconnection))
print("from_entity resolved:", icx_restored.from_entity.is_resolved(), "→", icx_restored.from_entity.get().name)
print("to_entity   resolved:", icx_restored.to_entity.is_resolved(), "→", icx_restored.to_entity.get().name)

from_entity resolved: True → SE4
to_entity   resolved: True → DK2


### Error modes

`Reference` that can't be resolved raises `UnresolvedReferenceError`. `Reference` pointing *outside* the tree raises on serialize.

In [12]:
from energydatamodel.reference import UnresolvedReferenceError

broken = edm.Portfolio(
    name="Broken",
    members=[edm.Interconnection(name="X", from_entity=edm.Reference("Ghost"), to_entity=edm.Reference("Ghost2"))],
)
try:
    edm.Portfolio.from_json(broken.to_json())
except UnresolvedReferenceError as e:
    print("caught:", e)

caught: Reference 'Ghost' does not resolve against Portfolio('Broken')


## 6. Powergrid tour

The powergrid types split by structural kind:

| Kind         | Classes                                                            |
|--------------|--------------------------------------------------------------------|
| `Node`       | `JunctionPoint`, `Meter`, `DeliveryPoint`, `Area`                  |
| `Edge`       | `Line`, `Link`, `Transformer`, `Interconnection`, `Pipe`           |
| `Collection` | `SubNetwork`, `Network`, `Site`, `Portfolio`, `Region`, `MultiSite`|

In [13]:
electricity = edm.Carrier(name="electricity", type="renewable")

bus_a = edm.JunctionPoint(name="BusA", carrier=electricity)
bus_b = edm.JunctionPoint(name="BusB", carrier=electricity)

line = edm.Line(
    name="A-B",
    from_entity=edm.Reference("BusA"),
    to_entity=edm.Reference("BusB"),
    capacity=400.0,
)

# Pipes use the same Edge contract — direction + capacity + medium.
gas = edm.Pipe(
    name="A-B-gas",
    from_entity=edm.Reference("BusA"),
    to_entity=edm.Reference("BusB"),
    capacity=50.0,
    medium="gas",
)

net = edm.Network(name="Mini-Grid", members=[bus_a, bus_b, line, gas])
print(net.to_tree())

Network('Mini-Grid')
├── JunctionPoint('BusA')
├── JunctionPoint('BusB')
├── Line('A-B')
└── Pipe('A-B-gas')


## 7. Tabular view

`to_properties()` returns the *domain-specific* fields on an asset (everything except base fields like `name`/`location`/`timeseries` and child lists). Handy for building dataframes or DB rows.

In [14]:
t01.to_properties()

{'capacity': 3.5, 'hub_height': 80}

---

That's the whole surface: `Element` as the root, `Node`/`Edge`/`Collection` as the three sibling subtrees, `Asset` as a cross-cutting mixin, plus `Reference[T]` for cross-tree links and an energy vocabulary for time series. See `tests/` for exhaustive usage; see `energydatamodel/json_io.py` if you need to register custom `Element` subclasses for JSON dispatch.